In [1]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split

random_seed = [42, 0, 4242, 1, 142]

In [ ]:
# Routes
csv_path = os.path.join("MCGLPPI_RawData", "PDBBINDdimer_strict_index.csv")
proaffinity_data_dir = os.path.join("ProAffinity-GNN", "data")


# Assurance
os.makedirs(proaffinity_data_dir, exist_ok=True)

In [3]:
# Read csv
print("Reading list...")
df = pd.read_csv(csv_path)

col_name = 'binding_affinity'

Reading list...


In [4]:
# 3. String to  "pKd (-log10(M))"
def calculate_pkd(affinity_str):
    if pd.isna(affinity_str):
        return np.nan
    
    # Regular Expression 
    match = re.search(r'([0-9.]+)\s*(fM|pM|nM|uM|mM|M)', str(affinity_str))
    if not match:
        return np.nan
    
    val = float(match.group(1))
    unit = match.group(2)
    
    # To standard Molar conc.
    unit_dict = {'fM': 1e-15, 'pM': 1e-12, 'nM': 1e-9, 'uM': 1e-6, 'mM': 1e-3, 'M': 1.0}
    molar_val = val * unit_dict[unit]
    
    #  -log10(M) 
    return -np.log10(molar_val)

df['proaffinity_label'] = df[col_name].apply(calculate_pkd)
df = df.dropna(subset=['proaffinity_label']) # drop

print(f" {len(df)} valid labels in total")

 1270 valid labels in total


In [ ]:
'''
# 4.  8:1:1 train/val/test
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Same random seed
train_df.to_csv("train_split.csv", index=False)
val_df.to_csv("val_split.csv", index=False)
test_df.to_csv("test_split.csv", index=False)
print("Same Train/Val/Test saved!")
'''

Same Train/Val/Test saved!


In [ ]:
split_dir = "random_seeds_splits"
os.makedirs(split_dir, exist_ok=True)

for seed in random_seed:
    seed_dir = os.path.join(split_dir, f"seed_{seed}")
    os.makedirs(seed_dir, exist_ok=True)
    
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=seed)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=seed)
    
    train_df.to_csv(os.path.join(seed_dir, "train_split.csv"), index=False)
    val_df.to_csv(os.path.join(seed_dir, "val_split.csv"), index=False)
    test_df.to_csv(os.path.join(seed_dir, "test_split.csv"), index=False)
    
    print(f" Random Seed {seed} is successfully assigned: {seed_dir}")

In [ ]:
# 5. ProAffinity-GNN exclusive TXT label files 
txt_kd_path = os.path.join(proaffinity_data_dir, "PPIdataindex_kd.txt")
txt_index_path = os.path.join(proaffinity_data_dir, "PPIdataindex.txt")

df.to_csv(txt_kd_path, columns=['pdb_code', 'proaffinity_label'], sep=' ', header=False, index=False)
df.to_csv(txt_index_path, columns=['pdb_code', 'proaffinity_label'], sep='\t', header=False, index=False)

print(f"ProAffinity exclusive list saved: {proaffinity_data_dir}")

ProAffinity exclusive list saved: ProAffinity-GNN\data
